# RNN Sentiment Analysis — IMDB

In [1]:
import re
from collections import Counter

from datasets import load_dataset

In [2]:
def tokenize(text):
    # lowercase, then pull out runs of word chars -> strips punctuation/HTML tags for free
    return re.findall(r"\b\w+\b", text.lower())

In [3]:
def build_vocab(texts, min_freq=2):
    '''
    This function takes whole text as input, tokenizes it and outputs a
    dict, that has the token as the key and the tokenid as the value.
    
    '''
    counts = Counter()
    for text in texts:
        counts.update(tokenize(text))

    vocab = {"<PAD>": 0, "<UNK>": 1}
    for word, freq in counts.items():
        if freq >= min_freq:
            vocab[word] = len(vocab)
    return vocab

In [4]:
def encode(text, vocab):
    return [vocab.get(token, vocab["<UNK>"]) for token in tokenize(text)]

In [5]:
imdb = load_dataset("stanfordnlp/imdb")
print(imdb)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [6]:
sample = imdb["train"][0]
print("label:", sample["label"], "(0 = neg, 1 = pos)")
print("text:", sample["text"][:300], "...")

label: 0 (0 = neg, 1 = pos)
text: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really h ...


In [7]:
vocab = build_vocab(imdb["train"]["text"])
print("vocab size:", len(vocab))

vocab size: 47010


In [8]:
encode(sample["text"], vocab)[:20]

[2, 3, 2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 18]

In [9]:
import torch

In [10]:
def pad_batch(encoded_reviews, pad_value=0):
    '''
    Right-pads a batch of variable-length encoded reviews to the batch's own max length.

    Returns:
        padded: (batch_size, max_len) torch.long tensor of token ids
        lengths: (batch_size,) torch.long tensor of each review's real (unpadded) length
    '''
    lengths = torch.tensor([len(r) for r in encoded_reviews], dtype=torch.long)
    max_len = lengths.max().item()

    padded = torch.full((len(encoded_reviews), max_len), pad_value, dtype=torch.long)
    for i, review in enumerate(encoded_reviews):
        padded[i, :len(review)] = torch.tensor(review, dtype=torch.long)

    return padded, lengths

In [11]:
test_reviews = [imdb["train"][i]["text"] for i in range(3)]
test_encoded = [encode(text, vocab) for text in test_reviews]
print("lengths before padding:", [len(r) for r in test_encoded])

padded, lengths = pad_batch(test_encoded)
print("padded shape:", padded.shape)
print("lengths:", lengths)
print(padded[:, :15])  # first 15 columns of each row, to eyeball the padding

lengths before padding: [297, 228, 97]
padded shape: torch.Size([3, 297])
lengths: tensor([297, 228,  97])
tensor([[  2,   3,   2,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,  14,
          15],
        [  2,   4,   5,   6,  54,  42, 181,  91, 182, 183, 184,  18, 177, 178,
         185],
        [ 33, 207,  36, 291,  75,  38, 292,  12, 155,  23,  14, 293,  38, 155,
          54]])


In [12]:
embed_dim = 100
embedding = torch.nn.Embedding(num_embeddings=len(vocab), embedding_dim=embed_dim, padding_idx=vocab["<PAD>"])

In [13]:
embedded = embedding(padded)
print("padded shape:", padded.shape)
print("embedded shape:", embedded.shape)

padded shape: torch.Size([3, 297])
embedded shape: torch.Size([3, 297, 100])


In [14]:
hidden_dim = 128
batch_size = padded.shape[0]

h = torch.zeros(batch_size, hidden_dim)
print("h shape:", h.shape)

h shape: torch.Size([3, 128])


In [15]:
t = 0
x_t = embedded[:, t, :]
print("x_t shape:", x_t.shape)

x_t shape: torch.Size([3, 100])


In [16]:
W_xh = (torch.randn(embed_dim, hidden_dim) * 0.01).requires_grad_()
W_hh = (torch.randn(hidden_dim, hidden_dim) * 0.01).requires_grad_()
b_h = torch.zeros(hidden_dim, requires_grad=True)

print("W_xh:", W_xh.shape, "W_hh:", W_hh.shape, "b_h:", b_h.shape)
print("leaf?", W_xh.is_leaf, W_hh.is_leaf, b_h.is_leaf)

W_xh: torch.Size([100, 128]) W_hh: torch.Size([128, 128]) b_h: torch.Size([128])
leaf? True True True


In [17]:
h = torch.tanh(x_t @ W_xh + h @ W_hh + b_h)
print("h shape after one step:", h.shape)

h shape after one step: torch.Size([3, 128])


In [18]:
max_len = padded.shape[1]
mask = (torch.arange(max_len).unsqueeze(0) < lengths.unsqueeze(1))
print("mask shape:", mask.shape)
print("real tokens per row:", mask.sum(dim=1))

mask shape: torch.Size([3, 297])
real tokens per row: tensor([297, 228,  97])


In [19]:
t = 150
x_t = embedded[:, t, :]
h_old = h
h_new = torch.tanh(x_t @ W_xh + h_old @ W_hh + b_h)

m = mask[:, t].unsqueeze(1)
h = torch.where(m, h_new, h_old)

print("mask at t:", mask[:, t])
print("row changed?", [not torch.equal(h[i], h_old[i]) for i in range(batch_size)])

mask at t: tensor([ True,  True, False])
row changed? [True, True, False]


In [27]:
h = torch.zeros(batch_size, hidden_dim)
hs = []

for t in range(max_len):
    x_t = embedded[:, t, :]
    h_new = torch.tanh(x_t @ W_xh + h @ W_hh + b_h)
    h = torch.where(mask[:, t].unsqueeze(1), h_new, h)
    hs.append(h)

print("steps stored:", len(hs), "| final h shape:", h.shape)

steps stored: 297 | final h shape: torch.Size([3, 128])


In [21]:
W_hy = (torch.randn(hidden_dim, 1) * 0.01).requires_grad_()
b_y = torch.zeros(1, requires_grad=True)

print("W_hy:", W_hy.shape, "b_y:", b_y.shape)

W_hy: torch.Size([128, 1]) b_y: torch.Size([1])


In [28]:
logits = h @ W_hy + b_y
print("logits shape:", logits.shape)
print(logits)

logits shape: torch.Size([3, 1])
tensor([[-0.0916],
        [-0.0837],
        [-0.0826]], grad_fn=<AddBackward0>)


In [23]:
y = torch.tensor([imdb["train"][i]["label"] for i in range(3)], dtype=torch.float32).unsqueeze(1)
print("y shape:", y.shape, "| y:", y.squeeze(1))

y shape: torch.Size([3, 1]) | y: tensor([0., 0., 0.])


In [29]:
loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, y)
print("loss:", loss.item())

loss: 0.6510948538780212


In [25]:
loss.backward()

for name, p in [("W_xh", W_xh), ("W_hh", W_hh), ("b_h", b_h), ("W_hy", W_hy), ("b_y", b_y)]:
    print(f"{name}: grad shape {tuple(p.grad.shape)} | mean |grad| = {p.grad.abs().mean().item():.2e}")

W_xh: grad shape (100, 128) | mean |grad| = 1.62e-03
W_hh: grad shape (128, 128) | mean |grad| = 1.54e-04
b_h: grad shape (128,) | mean |grad| = 3.81e-03
W_hy: grad shape (128, 1) | mean |grad| = 2.27e-02
b_y: grad shape (1,) | mean |grad| = 5.00e-01


In [26]:
lr = 0.1
params = [W_xh, W_hh, b_h, W_hy, b_y]

with torch.no_grad():
    for p in params:
        p -= lr * p.grad
        p.grad.zero_()

print("b_y after update:", b_y.item())

b_y after update: -0.05004967004060745


In [78]:
def forward(padded, lengths, training=False, p_drop=0.3):
    B, T = padded.shape
    embedded = embedding(padded)
    embedded = torch.nn.functional.dropout(embedded, p=p_drop, training=training)
    mask = torch.arange(T, device=padded.device).unsqueeze(0) < lengths.unsqueeze(1)

    h = torch.zeros(B, hidden_dim, device=padded.device)
    h_sum = torch.zeros(B, hidden_dim, device=padded.device)
    for t in range(T):
        m = mask[:, t].unsqueeze(1)
        h_new = torch.tanh(embedded[:, t, :] @ W_xh + h @ W_hh + b_h)
        h = torch.where(m, h_new, h)
        h_sum = h_sum + h * m          # only real steps contribute to the sum

    h_mean = h_sum / lengths.unsqueeze(1)
    return h_mean @ W_hy + b_y

In [79]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

max_tokens = 200
n_train = 22000
n_val = 2000
bs = 128
epochs = 40
lr = 0.2

torch.manual_seed(0)
perm = torch.randperm(25000)
subset = imdb["train"].select(perm[:n_train].tolist())
val_subset = imdb["train"].select(perm[n_train:n_train + n_val].tolist())

encoded = [encode(t, vocab)[-max_tokens:] for t in subset["text"]]
labels = torch.tensor(subset["label"], dtype=torch.float32).unsqueeze(1)
val_encoded = [encode(t, vocab)[-max_tokens:] for t in val_subset["text"]]
val_labels = torch.tensor(val_subset["label"], dtype=torch.float32).unsqueeze(1)

print("train positive fraction:", labels.mean().item(), "| val positive fraction:", val_labels.mean().item())
print("texts shared between train and val:", len(set(subset["text"]) & set(val_subset["text"])))

embedding = torch.nn.Embedding(len(vocab), embed_dim, padding_idx=vocab["<PAD>"]).to(device)
W_xh = (torch.randn(embed_dim, hidden_dim, device=device) * 0.01).requires_grad_()
W_hh = (torch.randn(hidden_dim, hidden_dim, device=device) / hidden_dim ** 0.5).requires_grad_()
b_h = torch.zeros(hidden_dim, device=device, requires_grad=True)
W_hy = (torch.randn(hidden_dim, 1, device=device) * 0.01).requires_grad_()
b_y = torch.zeros(1, device=device, requires_grad=True)

params = [embedding.weight, W_xh, W_hh, b_h, W_hy, b_y]
print("all leaf & on device:", all(p.is_leaf and p.device.type == device.type for p in params))

device: cuda
train positive fraction: 0.5018181800842285 | val positive fraction: 0.47850000858306885
texts shared between train and val: 12
all leaf & on device: True


In [80]:
def evaluate(encoded_reviews, labels, bs=128):
    total_loss, total_correct = 0.0, 0
    with torch.no_grad():
        for start in range(0, len(encoded_reviews), bs):
            padded_b, lengths_b = pad_batch(encoded_reviews[start:start + bs])
            padded_b, lengths_b = padded_b.to(device), lengths_b.to(device)
            y_b = labels[start:start + bs].to(device)

            logits = forward(padded_b, lengths_b)
            loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, y_b)
            total_loss += loss.item() * len(y_b)
            total_correct += ((logits > 0).float() == y_b).sum().item()
    return total_loss / len(encoded_reviews), total_correct / len(encoded_reviews)

In [ ]:
from tqdm.auto import tqdm

for epoch in range(epochs):
    order = torch.randperm(n_train)
    total_loss, total_correct = 0.0, 0
    for start in tqdm(range(0, n_train, bs), desc=f"epoch {epoch}"):
        idx = order[start:start + bs].tolist()
        padded_b, lengths_b = pad_batch([encoded[i] for i in idx])
        padded_b, lengths_b = padded_b.to(device), lengths_b.to(device)
        y_b = labels[idx].to(device)

        logits = forward(padded_b, lengths_b, training=True)
        loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, y_b)

        for p in params:
            p.grad = None
        loss.backward()
        with torch.no_grad():
            for p in params:
                p -= lr * p.grad

        total_loss += loss.item() * len(idx)
        total_correct += ((logits > 0).float() == y_b).sum().item()

    val_loss, val_acc = evaluate(val_encoded, val_labels)
    print(f"epoch {epoch}: train loss {total_loss / n_train:.4f} acc {total_correct / n_train:.3f} | val loss {val_loss:.4f} acc {val_acc:.3f}")

epoch 0:   0%|          | 0/172 [00:00<?, ?it/s]

epoch 0: train loss 0.3657 acc 0.835 | val loss 0.4485 acc 0.812


epoch 1:   0%|          | 0/172 [00:00<?, ?it/s]

epoch 1: train loss 0.3600 acc 0.840 | val loss 0.4444 acc 0.814


epoch 2:   0%|          | 0/172 [00:00<?, ?it/s]

...

epoch 7: train loss 0.3488 acc 0.843 | val loss 0.4767 acc 0.818


epoch 8:   0%|          | 0/172 [00:00<?, ?it/s]

epoch 8: train loss 0.3448 acc 0.849 | val loss 0.4422 acc 0.819


epoch 9:   0%|          | 0/172 [00:00<?, ?it/s]

epoch 9: train loss 0.3488 acc 0.845 | val loss 0.4370 acc 0.820


In [88]:
# FINAL HELD-OUT EVALUATION — run once, after all tuning is finished.
# imdb["test"] has never been touched, which is what makes this number honest.

test_encoded = [encode(t, vocab)[-max_tokens:] for t in imdb["test"]["text"]]
test_labels = torch.tensor(imdb["test"]["label"], dtype=torch.float32).unsqueeze(1)

test_loss, test_acc = evaluate(test_encoded, test_labels)
print(f"TEST: loss {test_loss:.4f} | acc {test_acc:.3f}  (n = {len(test_encoded)})")

TEST: loss 0.4220 | acc 0.819  (n = 25000)


In [ ]:
def predict(text):
    enc = encode(text, vocab)[-max_tokens:]
    if not enc:
        print("no tokens found")
        return None

    padded_b, lengths_b = pad_batch([enc])
    with torch.no_grad():
        logit = forward(padded_b.to(device), lengths_b.to(device))
        p = torch.sigmoid(logit).item()

    n_unk = sum(1 for i in enc if i == vocab["<UNK>"])
    verdict = "POSITIVE" if p > 0.5 else "NEGATIVE"
    print(f"{verdict}  p_pos={p:.3f}  |  {len(enc)} tokens used, {n_unk} unknown")
    return p




NEGATIVE  p_pos=0.000  |  21 tokens used, 0 unknown
NEGATIVE  p_pos=0.000  |  4 tokens used, 1 unknown


2.3876292609692484e-10

In [104]:
checkpoint = {
    # weights, in the same order as `params`
    "params": [p.detach().cpu() for p in params],
    "param_names": ["embedding.weight", "W_xh", "W_hh", "b_h", "W_hy", "b_y"],

    # without these the weights are meaningless
    "vocab": vocab,              # row i of the embedding table means whatever THIS vocab says
    "max_tokens": max_tokens,    # inputs must be truncated the same way
    "embed_dim": embed_dim,      # to rebuild the Embedding module
    "hidden_dim": hidden_dim,    # forward() reads this as a global

    # provenance
    "test_acc": test_acc,
    "test_loss": test_loss,
}

torch.save(checkpoint, "rnn_imdb.pt")
print(f"saved rnn_imdb.pt | {sum(p.numel() for p in params):,} params | test acc {test_acc:.3f}")

saved rnn_imdb.pt | 4,730,441 params | test acc 0.819
